## Notebook11

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
us_pop = (
    pl.read_csv(ub + "data/us_city_population.csv")
    .select(c.city, c.state, c.year, c.population, c.lon, c.lat)
    .filter(c.population.null_count().over(c.city) <= 1)
)

### Questions

This is the same table as last week: 289 American cities, one row per city per
census from 1790 to 2010, with the population in thousands. A city that did not
exist yet has a population of 0. The columns `lon` and `lat` give the location of
each city.

Today is a review for the exam on Wednesday. The questions go back over plots,
grouping, and window functions from the last few weeks, one step at a time.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Filter the data to the 2010 census and draw a scatterplot with longitude on the
x-axis and latitude on the y-axis.

2. Draw the same plot, but make all of the points red.

The color is fixed, so it goes inside `geom_point` but outside of `aes`.

3. Draw the same plot again, but now make the size of each point show the city's
population.

The size now comes from a column, so it goes inside `aes`.

4. Filter to the 2010 cities in Virginia (`"VA"`) and draw a text plot with longitude
on the x-axis, latitude on the y-axis, and the city name as the label.

5. Filter to Richmond (`"Richmond, VA"`) and draw a line plot of its population
against the year.

6. Summarize the whole dataset in a single row. Give the number of rows with
`pl.len()`, the number of different cities with `c.city.n_unique()`, and the number
of different states with `c.state.n_unique()`.

There are 6,647 rows but only 289 cities, because each city appears once for every
census. We use `.select` here because `.agg` only works after a `.group_by`.

7. For the 2010 census, group by state and compute the total population of the
cities in each state.

8. Take the code from the previous question and draw it as a bar plot with
`geom_col`, with the state on the x-axis and the total on the y-axis. Add
`coord_flip()` so the bars run horizontally.

California, Texas, and New York have the longest bars.

9. Group by year and count the number of cities with a population above zero in each
census. Use `(c.population > 0).sum()` inside of `.agg`. Sort by year.

Only 16 of the cities had any population in 1790. By 1900 there were 216.

10. Take the code from the previous question and draw it as a line plot against the
year.

11. Filter to three states, California, Texas, and New York, using
`c.state.is_in(["CA", "TX", "NY"])`. Then group by both state and year and compute
the total population.

12. Take the code from the previous question and draw a line plot of the total
against the year, with one color for each state.

13. For the 2010 census, sort by population from largest to smallest, and then use
`.group_by(c.state).head(n=1)` to keep the largest city in each state.

14. For the 2010 census, add a column `state_mean` giving the mean population of the
cities in the same state. Use `.over(c.state)`.

15. Now compute the same state means with `.group_by` and `.agg`. How is the result
different from the previous question?

**Answer**:


16. For the 2010 census, keep only the cities whose population is larger than the
mean population of the cities in their own state.

17. For the 2010 census, add a column `rank` giving each city's rank within its state,
with 1 for the largest. Use `c.population.rank(descending=True).over(c.state)`.

18. Sort by city and year, add a column `change` equal to
`c.population - c.population.shift(1).over(c.city)`, and filter to Richmond.

The first row is `null` because there is no census before 1790 to subtract.

19. Take the code from the previous question and draw a bar plot of `change` against
the year with `geom_col`.

Richmond grew in every decade until 1950 and has gone up and down since.

20. For the 2010 census, sort the cities from largest to smallest and add a column
`running` equal to `c.population.cum_sum()`.

Each row adds its population to everything above it, so the sort matters.

21. Sort by city and year and add a column `pop_smooth` equal to
`c.population.rolling_mean(window_size=3).over(c.city)`. Filter to Detroit
(`"Detroit, MI"`). Which rows are missing, and why?

**Answer**:


22. Last one. Compute `change` as in question 18, and then, in a second
`.with_columns`, add `change_smooth` equal to
`c.change.rolling_mean(window_size=3).over(c.city)`. Filter to Detroit.

The raw change jumps around: Detroit added 575,000 people in the 1920s and only
55,000 in the 1930s. The smoothed column rises and falls more gradually.